In [9]:
import napari
import tifffile
import numpy as np
from pathlib import Path
from magicgui.widgets import PushButton
from qtpy.QtCore import QTimer

In [10]:
folder = Path(r"C:\Users\taylorhearn\The University of Manchester Dropbox\Isobel Taylor-Hearn\masks_to_curate")
output_folder = Path(r"C:\Users\taylorhearn\The University of Manchester Dropbox\Isobel Taylor-Hearn\curated_masks")
output_folder.mkdir(parents=True, exist_ok=True)

images = sorted(folder.glob("*.tif"))
print(f"{len(images)} images found")

# Channel order within each tif (axis order is Z, C, Y, X)
BRIGHTFIELD = 0
FLUORESCENCE = 1
MASK = 2

14 images found


In [11]:
def read_image_and_meta(path):
    """Read a tif as a (Z, C, Y, X) array along with the metadata needed to
    write it back out unchanged (pixel size / spacing / unit)."""
    with tifffile.TiffFile(path) as tif:
        series = tif.series[0]
        arr = series.asarray()
        axes = series.axes
        ij_meta = dict(tif.imagej_metadata or {})
        page = tif.pages[0]

        def _rational(name):
            tag = page.tags.get(name)
            if tag is None:
                return None
            val = tag.value
            if isinstance(val, tuple) and len(val) == 2:
                return val[0] / val[1] if val[1] else None
            return val

        xres = _rational("XResolution")
        yres = _rational("YResolution")
        resunit_tag = page.tags.get("ResolutionUnit")
        resunit = int(resunit_tag.value) if resunit_tag is not None else None

    meta = {
        "axes": axes,
        "imagej": ij_meta,
        "xres": xres,
        "yres": yres,
        "resunit": resunit,
    }
    return arr, meta


def save_image(path, arr, meta):
    """Write `arr` to `path`, preserving the original pixel size / spacing /
    unit metadata."""
    ij = meta["imagej"]
    md = {"axes": meta["axes"]}
    for key in ("spacing", "unit", "finterval", "fps", "hyperstack",
                "mode", "channels", "slices", "frames"):
        if key in ij:
            md[key] = ij[key]

    kwargs = {"imagej": True, "metadata": md}
    if meta["xres"] and meta["yres"]:
        kwargs["resolution"] = (meta["xres"], meta["yres"])
    if meta["resunit"] is not None:
        kwargs["resolutionunit"] = meta["resunit"]

    tifffile.imwrite(path, arr, **kwargs)

In [12]:
# Curate only the middle z-slice of each tif. A single napari viewer is open
# at any moment, showing the middle slice of all three channels (BF, fluo,
# segmentation) in 2D. Edit the mask, then click "Save & Next" (or close the
# window):
#   - BF (c0) and fluo (c1) are kept unchanged (full z-stack)
#   - the curated middle-slice mask is duplicated across every z-slice to form
#     a prism, and written as the segmentation channel (c2)
# The result is saved as a new tif in `output_folder` with pixel-size metadata
# preserved. The viewer then reopens with the next image automatically.
#
# In a notebook the Qt event loop is already running, so we can't use a
# blocking `for` loop (that opens every viewer at once). Instead we chain
# images together through the viewer's close event.

class MaskCurator:
    def __init__(self, images, start_index=0):
        self.images = images
        self.index = start_index
        self.viewer = None
        self.arr = None
        self.meta = None
        self.labels_layer = None
        self.mid_z = None
        self._orig_close = None
        self._advancing = False

    def start(self):
        self._open_current()

    def _open_current(self):
        if self.index >= len(self.images):
            print("All images curated.")
            return

        path = self.images[self.index]
        print(f"[{self.index + 1}/{len(self.images)}] Curating {path.name}")

        self.arr, self.meta = read_image_and_meta(path)
        n_z = self.arr.shape[0]
        self.mid_z = n_z // 2

        # Middle slice of each channel (2D Y, X).
        brightfield = self.arr[self.mid_z, BRIGHTFIELD, :, :]
        fluorescence = self.arr[self.mid_z, FLUORESCENCE, :, :]
        mask = self.arr[self.mid_z, MASK, :, :]

        self.viewer = napari.Viewer(
            title=f"[{self.index + 1}/{len(self.images)}] {path.name} (z={self.mid_z})"
        )
        self.viewer.add_image(brightfield, name="brightfield",
                              colormap="gray", blending="additive")
        self.viewer.add_image(fluorescence, name="fluorescence",
                              colormap="green", blending="additive")
        self.labels_layer = self.viewer.add_labels(
            mask.astype(np.int32), name="mask"
        )

        next_btn = PushButton(text="Save & Next")
        next_btn.clicked.connect(self.viewer.close)
        self.viewer.window.add_dock_widget(next_btn, area="right",
                                           name="curation")

        # Route the window close (button or X) through our save+advance logic.
        qt_window = self.viewer.window._qt_window
        self._orig_close = qt_window.closeEvent
        qt_window.closeEvent = self._on_close

    def _on_close(self, event):
        if not self._advancing:
            self._advancing = True
            path = self.images[self.index]

            # Curated 2D middle slice -> duplicate across every z to make a prism.
            curated_slice = self.labels_layer.data.astype(self.arr.dtype)
            n_z = self.arr.shape[0]
            out = self.arr.copy()
            out[:, MASK, :, :] = np.broadcast_to(
                curated_slice, (n_z,) + curated_slice.shape
            )
            # BF (c0) and fluo (c1) remain unchanged in `out`.

            out_path = output_folder / path.name
            save_image(out_path, out, self.meta)
            print(f"    saved -> {out_path}")

            self.index += 1
            # Open the next image once this window has finished closing.
            QTimer.singleShot(200, self._open_next)
        self._orig_close(event)

    def _open_next(self):
        self._advancing = False
        self._open_current()


# Set start_index to resume part-way through the list if needed.
curator = MaskCurator(images, start_index=0)
curator.start()

[1/14] Curating 20260514_FL37_ARi_device1_infocus.tif


c:\Users\taylorhearn\AppData\Local\miniforge3\envs\nap-ij\Lib\site-packages\napari\_vispy\layers\scalar_field.py:197: UserWarning: data shape (1, 1674, 2602) exceeds GL_MAX_TEXTURE_SIZE 2048 in at least one axis and will be downsampled. Rendering is currently in 3D mode.
  warnings.warn(
c:\Users\taylorhearn\AppData\Local\miniforge3\envs\nap-ij\Lib\site-packages\napari\_vispy\layers\scalar_field.py:197: UserWarning: data shape (1, 1674, 2602) exceeds GL_MAX_TEXTURE_SIZE 2048 in at least one axis and will be downsampled. Rendering is currently in 3D mode.
  warnings.warn(


    saved -> C:\Users\taylorhearn\The University of Manchester Dropbox\Isobel Taylor-Hearn\curated_masks\20260514_FL37_ARi_device1_infocus.tif
[2/14] Curating 20260514_FL37_ARi_device4_infocus.tif


c:\Users\taylorhearn\AppData\Local\miniforge3\envs\nap-ij\Lib\site-packages\napari\_vispy\layers\scalar_field.py:197: UserWarning: data shape (1, 1714, 2589) exceeds GL_MAX_TEXTURE_SIZE 2048 in at least one axis and will be downsampled. Rendering is currently in 3D mode.
  warnings.warn(
c:\Users\taylorhearn\AppData\Local\miniforge3\envs\nap-ij\Lib\site-packages\napari\_vispy\layers\scalar_field.py:197: UserWarning: data shape (1, 2589, 1714) exceeds GL_MAX_TEXTURE_SIZE 2048 in at least one axis and will be downsampled. Rendering is currently in 3D mode.
  warnings.warn(
c:\Users\taylorhearn\AppData\Local\miniforge3\envs\nap-ij\Lib\site-packages\napari\_vispy\layers\scalar_field.py:197: UserWarning: data shape (1, 1714, 2589) exceeds GL_MAX_TEXTURE_SIZE 2048 in at least one axis and will be downsampled. Rendering is currently in 3D mode.
  warnings.warn(
c:\Users\taylorhearn\AppData\Local\miniforge3\envs\nap-ij\Lib\site-packages\napari\_vispy\layers\scalar_field.py:197: UserWarning: da

    saved -> C:\Users\taylorhearn\The University of Manchester Dropbox\Isobel Taylor-Hearn\curated_masks\20260514_FL37_ARi_device4_infocus.tif
[3/14] Curating 20260514_FL37_UTD_device2_infocus.tif
    saved -> C:\Users\taylorhearn\The University of Manchester Dropbox\Isobel Taylor-Hearn\curated_masks\20260514_FL37_UTD_device2_infocus.tif
[4/14] Curating 20260514_FL37_UTD_device4_infocus.tif
    saved -> C:\Users\taylorhearn\The University of Manchester Dropbox\Isobel Taylor-Hearn\curated_masks\20260514_FL37_UTD_device4_infocus.tif
[5/14] Curating 20260514_FL41_ARi_device1_infocus.tif
    saved -> C:\Users\taylorhearn\The University of Manchester Dropbox\Isobel Taylor-Hearn\curated_masks\20260514_FL41_ARi_device1_infocus.tif
[6/14] Curating 20260514_FL41_UTD_device1_infocus.tif
    saved -> C:\Users\taylorhearn\The University of Manchester Dropbox\Isobel Taylor-Hearn\curated_masks\20260514_FL41_UTD_device1_infocus.tif
[7/14] Curating 20260514_FL41_UTD_device3_infocus.tif
    saved -> C: